# Metrics CERCA

In [1]:
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [2]:
df_whole = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS.csv')

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv')
df

/tmp/ipykernel_5754/946240331.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv')


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,4.210155e+09,ES,False,ResearchMar
1,10.1038/s41591-023-02610-2,Charles Agyemang,47.0,middle,False,8.870644e+08,NL,False,ResearchMar
2,10.1038/s41591-023-02610-2,Andrew Wong,750.0,middle,False,4.512925e+07,GB,False,ResearchMar
3,10.1038/s41591-023-02610-2,Farhad Zamani,766.0,middle,False,1.611069e+08,IR,False,ResearchMar
4,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,6.363444e+07,ES,False,ResearchMar
...,...,...,...,...,...,...,...,...,...
193398,10.1111/jvh.13412,Karine Lacombe,33.0,middle,False,4.210134e+09,FR,False,ISGlobal
193399,10.1111/jvh.13412,Karine Lacombe,33.0,middle,False,4.210097e+09,FR,False,ISGlobal
193400,10.1111/jvh.13412,Karine Lacombe,33.0,middle,False,3.980408e+07,FR,False,ISGlobal
193401,10.1002/lol2.10277,Maria Lundgren,33.0,middle,False,2.234641e+08,SE,False,BETA


In [3]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA            146
CREAF           933
ICN2            828
ISGlobal       2744
ResearchMar    4452
dtype: int64

In [4]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             50
CREAF           825
ICN2            712
ISGlobal        626
ResearchMar    3725
dtype: int64

## Publications Number

In [5]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.7321995464852608


In [6]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             64
CREAF           914
ICN2            773
ISGlobal        727
ResearchMar    4070
dtype: int64

## % led publications

In [7]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.663718820861678


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [8]:
df_cerca = df[df.CERCA == True]
df_led = df_cerca[(df_cerca.author_position == 'first') | (df_cerca.author_position == 'last') |(df_cerca.is_corresponding == True) ]

df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.406250
CREAF          0.447484
ICN2           0.479948
ISGlobal       0.484182
ResearchMar    0.461425
dtype: float64

## \% publications with women from the centre as authors

In [9]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.663718820861678


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [10]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check.csv', index = False)
df_cerca

100%|██████████| 35846/35846 [00:00<00:00, 331771.79it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
296,10.1088/2515-7639/acc893,Masoud Karimipour,35.0,middle,False,4.210093e+09,ES,True,ICN2,Masoud,male
333,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,4.210091e+09,ES,True,ResearchMar,Silvia,female
334,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,4.210133e+09,ES,True,ResearchMar,Silvia,female
335,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,1.230449e+08,ES,True,ResearchMar,Silvia,female
485,10.1093/brain/awab362,Alessandro Príncipe,99.0,middle,False,NaN,NaN,True,ResearchMar,Alessandro,male
...,...,...,...,...,...,...,...,...,...,...,...
193141,10.1093/cid/ciab051,Quique Bassat,33.0,middle,False,7.199913e+07,ES,True,ISGlobal,Quique,male
193155,10.1111/exd.14291,Agustí Toll,33.0,middle,False,4.210131e+09,ES,True,ResearchMar,Agustí,unknown
193193,10.1016/j.cmi.2022.05.010,Juan Pablo Horcajada,33.0,middle,False,NaN,NaN,True,ResearchMar,Juan,male
193302,10.1038/s41380-022-01436-7,Carolina Minguillón,33.0,middle,False,4.210107e+09,ES,True,ResearchMar,Carolina,female


**We manually revise the classifier**

In [11]:
gender_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'Gender', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.1088/2515-7639/acc893,Masoud Karimipour,35.0,middle,False,4.210093e+09,ES,True,ICN2,Masoud,male,
1,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,4.210091e+09,ES,True,ResearchMar,Silvia,female,
2,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,4.210133e+09,ES,True,ResearchMar,Silvia,female,
3,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140.0,middle,False,1.230449e+08,ES,True,ResearchMar,Silvia,female,
4,10.1093/brain/awab362,Alessandro Príncipe,99.0,middle,False,NaN,NaN,True,ResearchMar,Alessandro,male,
...,...,...,...,...,...,...,...,...,...,...,...,...
35841,10.1093/cid/ciab051,Quique Bassat,33.0,middle,False,7.199913e+07,ES,True,ISGlobal,Quique,male,
35842,10.1111/exd.14291,Agustí Toll,33.0,middle,False,4.210131e+09,ES,True,ResearchMar,Agustí,male,male
35843,10.1016/j.cmi.2022.05.010,Juan Pablo Horcajada,33.0,middle,False,NaN,NaN,True,ResearchMar,Juan,male,
35844,10.1038/s41380-022-01436-7,Carolina Minguillón,33.0,middle,False,4.210107e+09,ES,True,ResearchMar,Carolina,female,


In [12]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.562500
CREAF          0.260394
ICN2           0.457956
ISGlobal       0.588721
ResearchMar    0.624816
dtype: float64

## \% publications led by women from the centre as authors

In [13]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.663718820861678


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [14]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.203125
CREAF          0.133479
ICN2           0.168176
ISGlobal       0.305365
ResearchMar    0.243735
dtype: float64

## \% publications in collaboration with other CERCA centres

In [15]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.663718820861678


In [16]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

for institution in df_cerca.Center.unique():
    df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
    df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
    percentage = df_colab.DOI.nunique() / df_center.DOI.nunique()
    print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ICN2 is: 21.05%
The percentage of publications in collaboration for ResearchMar is: 32.39%
The percentage of publications in collaboration for ISGlobal is: 21.69%
The percentage of publications in collaboration for CREAF is: 9.66%
The percentage of publications in collaboration for BETA is: 12.50%


In [17]:
# df_unique = df[['DOI', 'Center', 'CERCA']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# for institution in df_cerca.Center.unique():
#     dois_this = set(df_cerca.loc[df_cerca.Center == institution, 'DOI'])
#     dois_others = set(df_cerca.loc[df_cerca.Center != institution, 'DOI'])
#     collaborative_dois = dois_this & dois_others   
#     percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
#     print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

## \% publications in collaboration with other local institutions

In [18]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.7321995464852608


In [19]:
####### CREC QUE LES HE DE FER EXCLOENTS I A NIVELL DE DOI; AJUNTAR-LES

In [37]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

for institution in df_cerca.Center.unique():
    df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
    df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
    df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
    percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
    print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ICN2 is: 51.50%
The percentage of publications in collaboration for ResearchMar is: 63.50%
The percentage of publications in collaboration for ISGlobal is: 59.67%
The percentage of publications in collaboration for CREAF is: 46.76%
The percentage of publications in collaboration for BETA is: 75.00%


In [ ]:
# df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]
# for center in df_cerca['Center'].dropna().unique():
#     dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
#     dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
#     collaborative_dois = dois_center & dois_spanish_non_cerca
#     percentage = len(collaborative_dois) / len(dois_center)
#     print(f'The percentage of publications analyzed for {center} is: {percentage:.2%}')

The percentage of publications analyzed for ResearchMar is: 79.03%
The percentage of publications analyzed for ISGlobal is: 53.67%
The percentage of publications analyzed for BETA is: 88.00%
The percentage of publications analyzed for CREAF is: 39.88%
The percentage of publications analyzed for ICN2 is: 56.74%


## \% publications in collaboration with other international institutions

In [9]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

for center in df_cerca['Center'].dropna().unique():
    dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
    dois_international = set(df_international_non_cerca['DOI'])

    collaborative_dois = dois_center & dois_international
    
    percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
    print(f'The percentage of international collaborations for {center} is: {percentage:.2%}')

The percentage of international collaborations for ResearchMar is: 59.73%
The percentage of international collaborations for ISGlobal is: 78.27%
The percentage of international collaborations for BETA is: 58.00%
The percentage of international collaborations for CREAF is: 85.21%
The percentage of international collaborations for ICN2 is: 77.11%
